In [29]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [30]:
PROJECT_ROOT = Path.cwd().parent

RAW_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

PROCESSED_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data:")
print(RAW_PATH)

print("\nProcessed data:")
print(PROCESSED_PATH)

Project root:
d:\fraud-loss-analysis-step1\fraud-loss-analysis

Raw data:
d:\fraud-loss-analysis-step1\fraud-loss-analysis\data\raw

Processed data:
d:\fraud-loss-analysis-step1\fraud-loss-analysis\data\processed


In [31]:
customers = pd.read_csv(
    RAW_PATH / "customers.csv"
)

cards = pd.read_csv(
    RAW_PATH / "cards.csv"
)

merchants = pd.read_csv(
    RAW_PATH / "merchants.csv"
)

devices = pd.read_csv(
    RAW_PATH / "devices.csv"
)

transactions = pd.read_csv(
    RAW_PATH / "transactions.csv"
)

chargebacks = pd.read_csv(
    RAW_PATH / "chargebacks.csv"
)

print("Customers:", customers.shape)
print("Cards:", cards.shape)
print("Merchants:", merchants.shape)
print("Devices:", devices.shape)
print("Transactions:", transactions.shape)
print("Chargebacks:", chargebacks.shape)

Customers: (20000, 10)
Cards: (30000, 6)
Merchants: (2000, 6)
Devices: (35000, 3)
Transactions: (1501500, 14)
Chargebacks: (10500, 5)


In [32]:
transactions.head()

,txn_id,card_id,customer_id,merchant_id,device_id,txn_ts,amount,channel,category,merchant_city,merchant_country,ip_country,auth_result,customer_home_city
0,1,10359,6694,554,"16,884.00",2025-02-10 06:36:14,"2,253.14",E_COMMERCE,Fuel,Chandigarh,India,India,APPROVED,Ahmedabad
1,2,20959,9196,1643,"1,313.00",2025-12-07 15:12:17,"6,113.61",POS,Clothing,Kolkata,India,India,APPROVED,Chandigarh
2,3,3769,12095,500,"13,865.00",2025-08-24 23:11:07,"1,019.19",E_COMMERCE,Online Services,Mumbai,India,USA,APPROVED,Delhi
3,4,3632,12480,724,"32,906.00",2025-12-26 15:15:35,"1,447.45",E_COMMERCE,Entertainment,Chandigarh,India,India,APPROVED,Chennai
4,5,11165,2880,26,"29,838.00",2025-03-04 04:26:12,"1,534.02",E_COMMERCE,Entertainment,Ahmedabad,India,India,APPROVED,Jaipur


In [33]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 1501500 entries, 0 to 1501499
Data columns (total 14 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   txn_id              1501500 non-null  int64  
 1   card_id             1501500 non-null  int64  
 2   customer_id         1501500 non-null  int64  
 3   merchant_id         1501500 non-null  int64  
 4   device_id           1498496 non-null  float64
 5   txn_ts              1501500 non-null  str    
 6   amount              1501500 non-null  float64
 7   channel             1501500 non-null  str    
 8   category            1501500 non-null  str    
 9   merchant_city       1501500 non-null  str    
 10  merchant_country    1501500 non-null  str    
 11  ip_country          1498498 non-null  str    
 12  auth_result         1501500 non-null  str    
 13  customer_home_city  1501500 non-null  str    
dtypes: float64(2), int64(4), str(8)
memory usage: 160.4 MB


In [34]:
transactions.describe(include='all')

,txn_id,card_id,customer_id,merchant_id,device_id,txn_ts,amount,channel,category,merchant_city,merchant_country,ip_country,auth_result,customer_home_city
count,"1,501,500.00","1,501,500.00","1,501,500.00","1,501,500.00","1,498,496.00",1501500,"1,501,500.00",1501500,1501500,1501500,1501500,1498498,1501500,1501500
unique,NaN,NaN,NaN,NaN,NaN,1464813,NaN,4,11,15,5,5,2,16
top,NaN,NaN,NaN,NaN,NaN,2025-08-30 22:27:34,NaN,POS,Travel,Delhi,India,India,APPROVED,Delhi
freq,NaN,NaN,NaN,NaN,NaN,4,NaN,628188,162034,110054,1330199,1420495,1448945,109373
mean,"750,003.61","14,999.17","9,982.77","1,300.34","17,502.08",NaN,"2,508.65",NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,"433,007.29","8,655.99","5,770.17","17,301.55","10,104.82",NaN,"3,539.07",NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.00,1.00,1.00,1.00,1.00,NaN,"-51,211.37",NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,"375,018.75","7,506.00","5,003.00",501.00,"8,760.00",NaN,756.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,"749,992.50","14,996.00","9,941.00","1,001.00","17,504.00",NaN,"1,493.43",NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,"1,125,002.25","22,495.00","14,994.00","1,502.00","26,253.00",NaN,"2,948.92",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
def data_qualitty_report(df):

    report = pd.DataFrame({
        "column":df.columns,
        "dtype":df.dtypes.astype(str).values,
        "missing_count":df.isnull().sum().values,
        "missing_pct":(df.isnull().mean().values*100),
        "unique_values":[
            df[col].nunique(dropna=True) for col in df.columns
            ]
    })

    return report.sort_values( "missing_pct", ascending=False)

In [36]:
tranasctions_report = data_qualitty_report(transactions)
tranasctions_report

,column,dtype,missing_count,missing_pct,unique_values
4,device_id,float64,3004,0.20,35000
11,ip_country,str,3002,0.20,5
1,card_id,int64,0,0.00,30000
2,customer_id,int64,0,0.00,15548
3,merchant_id,int64,0,0.00,2001
0,txn_id,int64,0,0.00,1500000
5,txn_ts,str,0,0.00,1464813
6,amount,float64,0,0.00,554502
8,category,str,0,0.00,11
7,channel,str,0,0.00,4


In [37]:
transactions.duplicated().sum()

np.int64(1498)

In [38]:
duplicated_rows = transactions[
    transactions.duplicated(keep=False)
].sort_values("txn_id")

duplicated_rows.head(20)

,txn_id,card_id,customer_id,merchant_id,device_id,txn_ts,amount,channel,category,merchant_city,merchant_country,ip_country,auth_result,customer_home_city
1950,1951,5129,10957,660,"1,174.00",2025-11-08 15:35:07,"2,014.10",ATM,Electronics,Surat,India,India,APPROVED,Bengaluru
1501228,1951,5129,10957,660,"1,174.00",2025-11-08 15:35:07,"2,014.10",ATM,Electronics,Surat,India,India,APPROVED,Bengaluru
1500948,2815,16304,1805,1844,"30,440.00",2025-01-08 14:51:43,740.18,POS,Electronics,Bengaluru,India,India,APPROVED,Chandigarh
2814,2815,16304,1805,1844,"30,440.00",2025-01-08 14:51:43,740.18,POS,Electronics,Bengaluru,India,India,APPROVED,Chandigarh
3390,3391,28884,5775,838,"5,951.00",2025-02-15 07:17:29,847.61,CONTACTLESS,Jewelry,Surat,India,India,APPROVED,Chandigarh
1501209,3391,28884,5775,838,"5,951.00",2025-02-15 07:17:29,847.61,CONTACTLESS,Jewelry,Surat,India,India,APPROVED,Chandigarh
3509,3510,23147,1915,474,"29,019.00",2025-11-11 10:36:42,"1,752.04",POS,Grocery,Surat,India,India,APPROVED,Bengaluru
1500121,3510,23147,1915,474,"29,019.00",2025-11-11 10:36:42,"1,752.04",POS,Grocery,Surat,India,India,APPROVED,Bengaluru
1500197,4668,24036,11011,394,"21,615.00",2025-09-11 18:59:21,471.99,POS,Dining,Indore,India,India,APPROVED,Pune
4667,4668,24036,11011,394,"21,615.00",2025-09-11 18:59:21,471.99,POS,Dining,Indore,India,India,APPROVED,Pune


In [39]:
duplicated_rows.shape

(2996, 14)

In [40]:
invalid_txn = transactions[transactions["amount"] < 0 ]

invalid_txn.shape

(449, 14)

In [41]:
invalid_txn[
    [
        "txn_id",
        "card_id",
        "merchant_id",
        "txn_ts",
        "amount",
        "channel"
    ]
].head(20)

,txn_id,card_id,merchant_id,txn_ts,amount,channel
8997,8998,12321,443,2025-04-02 18:25:11,"-51,211.37",POS
10451,10452,8932,1688,2025-05-17 19:30:58,"-11,224.60",POS
11855,11856,7606,747,2025-01-30 15:27:22,"-1,984.20",E_COMMERCE
13146,13147,13124,1609,2025-10-16 03:56:46,"-1,903.08",ATM
16238,16239,22226,903,2025-05-01 13:10:15,-876.79,E_COMMERCE
19481,19482,520,476,2025-10-28 21:17:09,-527.75,E_COMMERCE
21786,21787,22548,1212,2025-07-19 05:05:13,"-6,276.35",POS
21797,21798,13722,149,2025-05-16 08:28:15,-896.08,POS
22994,22995,13707,731,2025-09-08 12:01:02,"-1,130.17",POS
35664,35665,13432,947,2025-05-18 19:02:56,"-3,271.06",ATM


In [42]:
valid_merchant_ids = set(
    merchants["merchant_id"]
)

In [43]:
orphan_txn = transactions[
    ~transactions["merchant_id"].isin(
        valid_merchant_ids
    )
]

print("orpan transactions",len(orphan_txn))

orpan transactions 450


In [44]:
orphan_txn[
    [
        "txn_id",
        "merchant_id",
        "amount",
        "channel",
        "category"
    ]
].head()

,txn_id,merchant_id,amount,channel,category
2968,2969,999999,776.89,POS,Jewelry
3646,3647,999999,"2,485.96",POS,Online Services
8007,8008,999999,"4,686.94",E_COMMERCE,Jewelry
8733,8734,999999,685.60,E_COMMERCE,Electronics
10108,10109,999999,"6,986.98",POS,Entertainment


In [45]:
transactions[
    "customer_home_city"
].value_counts()

customer_home_city
Delhi         109373
Chennai       104102
Lucknow       103824
Pune          102710
Chandigarh    102444
Kolkata       101292
Bangalore     100491
Indore         99701
Jaipur         99550
Hyderabad      99089
Surat          97659
Nagpur         97261
Mumbai         96188
Bengaluru      94748
Ahmedabad      92773
BLR              295
Name: count, dtype: int64

In [46]:
CITY_MAPPING ={
    "Bangalore":"Bengaluru",
    "BLR":"Bengaluru"
}

transactions[
    "customer_home_city"
] = transactions["customer_home_city"].replace(CITY_MAPPING)

In [47]:
transactions[
    "customer_home_city"
].value_counts()

customer_home_city
Bengaluru     195534
Delhi         109373
Chennai       104102
Lucknow       103824
Pune          102710
Chandigarh    102444
Kolkata       101292
Indore         99701
Jaipur         99550
Hyderabad      99089
Surat          97659
Nagpur         97261
Mumbai         96188
Ahmedabad      92773
Name: count, dtype: int64

In [48]:
transactions["txn_ts"] = pd.to_datetime(
    transactions["txn_ts"],
    errors="coerce",
    format="mixed"
)

In [51]:
print("Timestamp datatype:")
print(transactions["txn_ts"].dtype)

print("\nInvalid timestamps:")
print(transactions["txn_ts"].isna().sum())

print("\nMinimum timestamp:")
print(transactions["txn_ts"].min())

print("\nMaximum timestamp:")
print(transactions["txn_ts"].max())

Timestamp datatype:
datetime64[us]

Invalid timestamps:
0

Minimum timestamp:
2025-01-01 00:00:03

Maximum timestamp:
2025-12-31 23:59:35


In [53]:
missing_summary = (
    transactions.isna().sum().sort_values(ascending=False)
)

missing_summary[missing_summary>0]

device_id     3004
ip_country    3002
dtype: int64

In [54]:
transactions["ip_country"] = (
    transactions["ip_country"].fillna("UNKNOWN")
)

In [55]:
transactions["device_id"] = (
    transactions["device_id"].fillna(-1).astype(int)
)

In [57]:
transactions[["ip_country","device_id"]].isna().sum()

ip_country    0
device_id     0
dtype: int64

In [58]:
duplicate_count = transactions.duplicated().sum()

print("Duplicated rows before deleting",duplicate_count)

Duplicated rows before deleting 1498


In [59]:
transactions = transactions.drop_duplicates(
    keep="first"
).reset_index(
    drop=True
)

In [60]:
duplicate_count = transactions.duplicated().sum()

print("Duplicated rows after deleting",duplicate_count)

Duplicated rows after deleting 0


In [62]:
invalid_amnt_count = (
    transactions["amount"] <=0
).sum()

print("invalid amounts",invalid_amnt_count)

invalid amounts 1199


In [63]:
transactions = transactions[
    transactions["amount"] > 0
].copy()

In [64]:
transactions = transactions.reset_index(
    drop=True
)

In [65]:
(
    transactions["amount"] <= 0
).sum()

np.int64(0)

In [66]:
orphan_count = (
    ~transactions["merchant_id"]
    .isin(valid_merchant_ids)
).sum()

print(
    "Orphan merchant transactions:",
    orphan_count
)

Orphan merchant transactions: 450


In [67]:
transactions = transactions[
    transactions["merchant_id"].isin(
        valid_merchant_ids
    )
].copy()

In [68]:
transactions = transactions.reset_index(
    drop=True
)

In [69]:
(
    ~transactions["merchant_id"]
    .isin(valid_merchant_ids)
).sum()

np.int64(0)

In [70]:
print("Rows:", len(transactions))

print(
    "Duplicate rows:",
    transactions.duplicated().sum()
)

print(
    "Invalid amounts:",
    (transactions["amount"] <= 0).sum()
)

print(
    "Missing device:",
    transactions["device_id"].isna().sum()
)

print(
    "Missing IP:",
    transactions["ip_country"].isna().sum()
)

print(
    "Orphan merchants:",
    (
        ~transactions["merchant_id"]
        .isin(valid_merchant_ids)
    ).sum()
)

Rows: 1498353
Duplicate rows: 0
Invalid amounts: 0
Missing device: 0
Missing IP: 0
Orphan merchants: 0


In [72]:
quality_log = pd.DataFrame({
    "issue": [
        "Duplicate transactions",
        "Invalid transaction amounts",
        "Missing device IDs",
        "Missing IP country",
        "Orphan merchant IDs",
        "Inconsistent city names"
    ],

    "count": [
        duplicate_count,
        invalid_amnt_count,
        3004,
        3002,
        orphan_count,
        0
    ],

    "action": [
        "Removed exact duplicates",
        "Removed amount <= 0",
        "Replaced with -1 (Unknown)",
        "Replaced with UNKNOWN",
        "Removed orphan transactions",
        "Standardized to Bengaluru"
    ]
})

quality_log

,issue,count,action
0,Duplicate transactions,0,Removed exact duplicates
1,Invalid transaction amounts,1199,Removed amount <= 0
2,Missing device IDs,3004,Replaced with -1 (Unknown)
3,Missing IP country,3002,Replaced with UNKNOWN
4,Orphan merchant IDs,450,Removed orphan transactions
5,Inconsistent city names,0,Standardized to Bengaluru


In [73]:
transactions.to_csv(
    PROCESSED_PATH / "fact_transaction_clean.csv",
    index=False
)

In [74]:
quality_log.to_csv(
    PROCESSED_PATH / "data_quality_log.csv",
    index=False
)

In [75]:
customers.to_csv(
    PROCESSED_PATH / "dim_customer.csv",
    index=False
)

cards.to_csv(
    PROCESSED_PATH / "dim_card.csv",
    index=False
)

merchants.to_csv(
    PROCESSED_PATH / "dim_merchant.csv",
    index=False
)

devices.to_csv(
    PROCESSED_PATH / "dim_device.csv",
    index=False
)

chargebacks.to_csv(
    PROCESSED_PATH / "fact_chargeback.csv",
    index=False
)

In [76]:
print("=" * 60)
print("CLEANING COMPLETE")
print("=" * 60)

print(
    f"Clean transactions: {len(transactions):,}"
)

print(
    f"Customers: {len(customers):,}"
)

print(
    f"Cards: {len(cards):,}"
)

print(
    f"Merchants: {len(merchants):,}"
)

print(
    f"Devices: {len(devices):,}"
)

print("\nProcessed files:")

for file in PROCESSED_PATH.iterdir():
    print(
        f" - {file.name}"
    )

CLEANING COMPLETE
Clean transactions: 1,498,353
Customers: 20,000
Cards: 30,000
Merchants: 2,000
Devices: 35,000

Processed files:
 - data_quality_log.csv
 - dim_card.csv
 - dim_customer.csv
 - dim_device.csv
 - dim_merchant.csv
 - fact_chargeback.csv
 - fact_transaction_clean.csv
